In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
import copy
import json
import os
import pandas as pd
import z_schemas as z_s

# JSON Functions

def readStateJson(pipeline_json):
    with open(pipeline_json, mode="r") as file:
        state = json.load(file)
    copy_state = copy.deepcopy(state)
    return state, copy_state

def updateStateJson(updated_state, pipeline_json):
    temp_file = pipeline_json.replace(".json", ".tmp")
    with open(temp_file, mode="w") as file:
        json.dump(updated_state, file, indent=4)
    os.replace(temp_file, pipeline_json)

# Test Functions

def check_field_presence(df, filename):
    print("Running the 'check_field_presence' test...")
    schema = z_s.SCHEMAS[filename]["fields"]
    schema_fields = list(schema.keys())
    if "data_date" in schema_fields:
        schema_fields.remove("data_date")
    file_fields = df.columns
    # The missing_fields list is designed to start off as a copy of the schema_fields list
    # As the code progresses, items are removed from this list and the list should ultimately end up empty
    # If it doesn't, this means a field is "missing" from the new file that occurs in the database table
    missing_fields = copy.deepcopy(schema_fields)
    new_fields = []
    for field in file_fields:
        if field in schema_fields:
            missing_fields.remove(field)
        else:
            new_fields.append(field)
    if (not missing_fields) and (not new_fields):
        print("Test passed.")
        return True
    else:
        print(f"The fields of the new file are different than the schema of the database table.")
        if missing_fields:
            print(f"The following fields are missing: {missing_fields}.")
        if new_fields:
            print(f"The following fields are new: {new_fields}.")
        return False

# 'filename' is unused in the following function, but I'm keeping it for signature consistency
def check_field_uniqueness(df, filename, field):
    print("Running the 'check_field_uniqueness' test...")
    count = df.count()
    dedup_count = df.dropDuplicates(subset=[field]).count()
    if dedup_count == count:
        print("Test passed.")
        return True
    else:
        print(f"The field '{field}' contains {count - dedup_count} duplicate value(s).")
        return False

# 'filename' is unused in the following function, but I'm keeping it for signature consistency
def check_no_fully_null_columns(df, filename):
    print("Running the 'check_no_fully_null_column' test..")
    count = df.count()
    cols = df.columns
    fully_null_cols = []

    for col in cols:
        # df.column_name.isNull() only works if you type the column name directly.
        # F.col(col).isNull() works when the column name comes from a variable (like this loop).
        # Use F.col() whenever the column name isn't hardcoded.
        col_null_count = df.filter(F.col(col).isNull()).count()
        if col_null_count == count:
            fully_null_cols.append(col)
    
    if fully_null_cols:
        print(f"Fully-null columns detected: {fully_null_cols}")
        return False
    print("Test passed.")
    return True

# This is much slower than I would like
# Will revisit this at some point
# This ran significantly faster when written for pandas
def check_dtype_consistency(df, filename):
    print("Running the 'check_dtype_consistency' test...")
    schema = z_s.SCHEMAS[filename]["fields"]
    inconsistent_fields = []
    for field, dtype in schema.items():
        if field == "data_date" or dtype == "TEXT":
            continue

        null_count = df.filter(F.col(field).isNull()).count()

        if dtype == "DATE":
            # casted_df = df.select(F.col(field).try_cast(DateType()).alias(field))
            casted_df = df.select(F.try_to_date(F.col(field), "MM/dd/yyyy").alias(field))
        elif dtype == "INTEGER":
            casted_df = df.select(F.col(field).try_cast(IntegerType()).alias(field))
        else:
            print(f"Unaccounted datatype '{dtype}' encountered for the '{field}' field. Review and update code.")
            inconsistent_fields.append(field)
            continue

        casted_null_count = casted_df.filter(F.col(field).isNull()).count()
   
        if casted_null_count != null_count:
            inconsistent_fields.append(field)
    if inconsistent_fields:
        print(f"Test failed on the following field(s):")
        for field in inconsistent_fields:
            print(f"{field}")
        return False
    else:
        print("Test passed.")
        return True
